# Kubernetes-Netzwerke – praktische Experimente

Wir verwenden **zwei Pods ohne Deployment und ohne Service**:

- `network-multi`: NGINX mit der Antwort **Pod A** und eine Alpine-Toolbox mit `curl` und `nc` (Netcat).
- `network-nginx`: NGINX mit der Antwort **Pod B**.

Die Toolbox installiert `curl` und `netcat-openbsd` beim Containerstart per `apk`. Dafür benötigt der Cluster Zugriff auf die Alpine-Paketquellen.

Das Notebook verwendet den Python-Kernel mit `%%bash`-Codezellen. Die YAML-Dateien liegen neben dem Notebook. Die Übungen laufen auf dem Deskmate mit Zugriff auf K3s.

**Lernziele:** gemeinsamer Netzwerk-Namespace, Pod-IP, Node-IP, DNS-Konfiguration und Hostzugriff unterscheiden.

## 1. Verbindung und Namespace prüfen

Wir arbeiten im Namespace `default`. Zuerst prüfen wir die Nodes und den Kontext.

In [ ]:
%%bash
kubectl config current-context
kubectl config set-context --current --namespace=default
kubectl get nodes -o wide


## 2. Die Pod-Manifeste ansehen

Im ersten Pod teilen sich NGINX und Toolbox denselben Netzwerk-Namespace. Die beiden Webserver liefern **unterschiedliche HTML-Texte**: Pod A bzw. Pod B. Damit lässt sich die Herkunft jeder Antwort direkt erkennen.

In [ ]:
%%bash
cat network-multi.yaml
echo '--- Zweiter Pod ---'
cat network-nginx.yaml


## 3. Beide Pods bereitstellen

Wir verwenden ausschließlich `kubectl apply`. Anschließend warten wir, bis die Pods bereit sind.

In [ ]:
%%bash
kubectl apply -f network-multi.yaml
kubectl apply -f network-nginx.yaml
kubectl wait --for=condition=Ready pod/network-multi --timeout=180s
kubectl wait --for=condition=Ready pod/network-nginx --timeout=180s


## 4. Pod-IP und Node-Zuordnung bestimmen

**Pod-IPs** werden dem Pod-Netzwerk zugeordnet; jeder unserer Pods besitzt eine eigene IP. **Node-IPs** gehören dagegen zu den Kubernetes-Knoten. Die Beispiel-IPs sind dynamisch – wir lesen die tatsächlichen Werte aus.

In [ ]:
%%bash
kubectl get pods -o wide
kubectl get nodes -o wide


Die IP-Adressen lassen sich auch gezielt per JSONPath auslesen. Die Pod-IP ist **nicht** die IP eines einzelnen Containers: Beide Container in `network-multi` teilen sich diese Adresse.

In [ ]:
%%bash
echo -n 'network-multi Pod-IP: '
kubectl get pod network-multi -o jsonpath='{.status.podIP}'; echo
echo -n 'network-nginx Pod-IP: '
kubectl get pod network-nginx -o jsonpath='{.status.podIP}'; echo
echo 'Node-Adressen:'
kubectl get nodes -o jsonpath='{range .items[*]}{.metadata.name}{"\t"}{range .status.addresses[*]}{.type}={.address}{" "}{end}{"\n"}{end}'


## 5. Interaktiv mit `kubectl exec` in die Toolbox

**Im VS-Code-Terminal** (nicht als Notebook-Zelle) eine interaktive Shell starten:

```bash
kubectl exec -it network-multi -c toolbox -- /bin/sh
```

Dort können wir `curl http://localhost`, `hostname`, `cat /etc/hosts`, `cat /etc/resolv.conf` und `nc -h` ausprobieren. Mit `exit` verlassen wir die Shell.

Die folgenden Notebook-Zellen zeigen dieselben Versuche nichtinteraktiv.

## 6. Container → Container im selben Pod: `localhost`

NGINX und Toolbox teilen sich **eine Pod-IP, das Loopback-Interface und denselben Portbereich**. Der Aufruf von `localhost:80` aus der Toolbox muss die Antwort **Pod A** liefern. Ein Service ist nicht nötig.

In [ ]:
%%bash
kubectl exec network-multi -c toolbox -- curl -sS http://127.0.0.1:80/


Auch die Pod-IP des **eigenen Pods** funktioniert. Die Anfrage bleibt damit innerhalb des Pod-Netzwerks.

In [ ]:
%%bash
POD_IP=$(kubectl get pod network-multi -o jsonpath='{.status.podIP}')
echo "Eigene Pod-IP: $POD_IP"
kubectl exec network-multi -c toolbox -- curl -sS "http://$POD_IP:80/"


## 7. Pod → anderer Pod: Pod-IP

Jetzt greift die Toolbox auf NGINX im **anderen** Pod zu. Wir erwarten die Antwort **Pod B**. `localhost` dagegen würde weiterhin Pod A erreichen.

In [ ]:
%%bash
OTHER_POD_IP=$(kubectl get pod network-nginx -o jsonpath='{.status.podIP}')
echo "Ziel-Pod-IP: $OTHER_POD_IP"
kubectl exec network-multi -c toolbox -- curl -sS "http://$OTHER_POD_IP:80/"


### 7.1 Drei Wege direkt vergleichen

Die drei Befehle laufen **alle aus derselben Toolbox**: `localhost`, eigene Pod-IP und fremde Pod-IP. Die ersten beiden Antworten müssen **Pod A**, die dritte **Pod B** zeigen.

In [ ]:
%%bash
OWN_IP=$(kubectl get pod network-multi -o jsonpath='{.status.podIP}')
OTHER_IP=$(kubectl get pod network-nginx -o jsonpath='{.status.podIP}')
echo '1) localhost:'
kubectl exec network-multi -c toolbox -- curl -sS http://localhost:80/
echo '2) eigene Pod-IP:'
kubectl exec network-multi -c toolbox -- curl -sS "http://$OWN_IP:80/"
echo '3) andere Pod-IP:'
kubectl exec network-multi -c toolbox -- curl -sS "http://$OTHER_IP:80/"


Das Pod-Netzwerk ermöglicht die Kommunikation grundsätzlich auch **über Node-Grenzen hinweg**. Ob die beiden Pods auf unterschiedlichen Nodes liegen, sehen wir mit dem folgenden Befehl. Die Platzierung hängt vom Scheduler ab und ist nicht garantiert.

In [ ]:
%%bash
kubectl get pods network-multi network-nginx -o custom-columns=NAME:.metadata.name,POD_IP:.status.podIP,NODE:.spec.nodeName


## 8. DNS-Konfiguration und Namensauflösung

Ein Pod bekommt standardmäßig eine **DNS-Konfiguration**. `/etc/resolv.conf` enthält den Nameserver und die Suchdomänen. Das bedeutet **nicht**, dass jeder Pod automatisch über seinen Namen im Cluster-DNS auflösbar ist.

Die lokale Datei `/etc/hosts` enthält dagegen üblicherweise den eigenen Pod-Hostnamen und die Pod-IP. **Ein Eintrag in `/etc/hosts` ist kein DNS-Record.**

Einen stabilen, clusterweit auflösbaren Namen erhalten Anwendungen üblicherweise über einen **Service**. Wir erstellen in dieser Übung bewusst keinen Service.

In [ ]:
%%bash
kubectl exec network-multi -c toolbox -- cat /etc/resolv.conf


### 8.1 Lokale Namenseinträge in `/etc/hosts`

Kubernetes trägt die Pod-IP und den Hostnamen in die lokale Datei `/etc/hosts` ein. Dies ermöglicht die Namensauflösung innerhalb des Pods, ohne einen DNS-Server zu kontaktieren. Der folgende `cat`-Befehl zeigt diese Zuordnung.

In [ ]:
%%bash
kubectl exec network-multi -c toolbox -- cat /etc/hosts


### 8.2 Service-DNS im Gegensatz zum Pod-Namen

Der bereits vorhandene Kubernetes-API-Service hat den DNS-Namen `kubernetes.default.svc.cluster.local`. Wir prüfen, dass der Name aufgelöst werden kann. Der HTTP-/TLS-Status der API ist für diesen Test nicht entscheidend.

`network-nginx` ist hingegen **nur der Pod-Name**, kein Service. Ohne zusätzliche DNS-Konfiguration ist dieser Name normalerweise nicht auflösbar. Die Service-DNS-Namenskonvention behandeln wir später ausführlich im Service-Kapitel.

In [ ]:
%%bash
kubectl exec network-multi -c toolbox -- sh -c 'curl -vk --connect-timeout 5 --max-time 8 https://kubernetes.default.svc.cluster.local/ 2>&1 | head -n 18'


**Erwartetes Negativbeispiel:** Der Name des zweiten Pods ist kein Service-Name. Wir fangen den erwarteten Fehler ab, damit das Notebook weiterläuft.

In [ ]:
%%bash
kubectl exec network-multi -c toolbox -- sh -c 'curl -sS --connect-timeout 3 --max-time 5 http://network-nginx/ || echo "Erwartet: kein DNS-Eintrag für den bloßen Pod-Namen"'


## 9. Einen TCP-Listener in der Toolbox starten: `nc`

**Verständnisfrage:** Können NGINX und Toolbox im selben Pod beide auf Port 80 lauschen?

Wir testen zunächst einen **freien Port 9090**. `nc -l -p 9090` startet einen TCP-Listener. In einer einzigen Notebook-Zelle starten wir ihn im Hintergrund, verbinden uns von **Pod B** aus und beenden ihn anschließend. Weil der zweite Pod kein `curl` enthält, nutzen wir für den Verbindungsversuch `kubectl exec` in der Toolbox und die **eigene Pod-IP**. Der Listener ist dabei aus jedem Container des Pods unter derselben IP erreichbar.

In [ ]:
%%bash
set -e
kubectl exec network-multi -c toolbox -- sh -c 'nc -l -p 9090 >/tmp/netcat-9090.log 2>&1 & echo $! >/tmp/netcat-9090.pid'
sleep 1
POD_IP=$(kubectl get pod network-multi -o jsonpath='{.status.podIP}')
echo "Verbindung zu $POD_IP:9090"
kubectl exec network-multi -c toolbox -- sh -c "printf 'Hallo auf Port 9090\n' | nc -w 3 $POD_IP 9090"
kubectl exec network-multi -c toolbox -- sh -c 'cat /tmp/netcat-9090.log; kill $(cat /tmp/netcat-9090.pid) 2>/dev/null || true'


### 9.1 Portkonflikt mit NGINX demonstrieren

NGINX belegt bereits Port 80 **im gemeinsamen Netzwerk-Namespace**. Ein weiterer TCP-Listener auf derselben Adresse und Portnummer sollte deshalb mit `Address already in use` scheitern. Der Befehl wird mit `timeout` begrenzt, damit keine Notebook-Zelle dauerhaft blockiert. Das Experiment läuft innerhalb des Toolbox-Containers; NGINX bleibt unverändert.

In [ ]:
%%bash
kubectl exec network-multi -c toolbox -- sh -c 'timeout 3 nc -l -p 80 2>&1; status=$?; echo "Exit-Code: $status (bei Portkonflikt ungleich 0)"'


**Erklärung:** Der Port gehört nicht exklusiv zu einem Container, sondern zum gemeinsam genutzten Netzwerk-Namespace des Pods. Deshalb können NGINX und Toolbox nicht beide auf derselben IP und demselben TCP-Port lauschen. Ein Listener auf einem **anderen Port** (9090) ist dagegen möglich.

## 9. Node-IP ist nicht Pod-IP

Ein Node besitzt eine eigene IP-Adresse im Node-Netzwerk. Diese ist **kein Ersatz für die Pod-IP**. Ohne NodePort, HostPort oder eine andere Veröffentlichung wird der NGINX-Port 80 des Pods **nicht automatisch** auf Port 80 des Nodes bereitgestellt.

Wir lesen beide Adresstypen aus und vergleichen sie. Einen Zugriff auf die Node-IP:80 führen wir absichtlich nicht als positiven Test aus – dort könnte ein ganz anderer Dienst laufen.

In [ ]:
%%bash
echo 'Pods:'
kubectl get pods -o wide
echo 'Nodes:'
kubectl get nodes -o wide


## 10. Ubuntu-Host → Pod: Port-Forwarding

Die Pod-IP ist vom Deskmate aus **nicht zwingend direkt erreichbar**. Mit `kubectl port-forward` können wir den Pod unabhängig von einer externen Node-IP testen. Für die Notebook-Ausführung starten wir Port-Forwarding kurzzeitig im Hintergrund und beenden es automatisch beim Verlassen der Zelle.

`18080:80` bedeutet: **localhost:18080 auf dem Deskmate → Port 80 im Pod**.

In [ ]:
%%bash
set -e
kubectl port-forward pod/network-nginx 18080:80 >/tmp/network-nginx-portforward.log 2>&1 &
pid=$!
trap 'kill "$pid" 2>/dev/null || true' EXIT
sleep 3
curl --max-time 10 -sS -I http://127.0.0.1:18080/


Dasselbe funktioniert für den NGINX-Container im Pod `network-multi`. Obwohl dort zwei Container laufen, besitzt der Pod nur einen Netzwerk-Namespace.

In [ ]:
%%bash
set -e
kubectl port-forward pod/network-multi 18080:80 >/tmp/network-multi-portforward.log 2>&1 &
pid=$!
trap 'kill "$pid" 2>/dev/null || true' EXIT
sleep 3
curl --max-time 10 -sS -I http://127.0.0.1:18080/


## 11. Abschlussbeispiel: Pod über die Node-IP erreichen (`hostPort`)

Bisher war der NGINX-Port **nur über die Pod-IP** erreichbar. Wir erzeugen jetzt einen **dritten Pod C**, dessen NGINX-Port 80 über `hostPort: 8080` an den **Node-Port 8080 des ausführenden Nodes** gebunden wird. Dazu benötigen wir weiterhin **keinen Service**.

**Wichtig für unsere K3s-in-Docker-Umgebung:** Die Node-IP gehört zu einem Docker-Container. Von einem anderen Netzwerk aus ist sie möglicherweise nicht erreichbar, obwohl `hostPort` korrekt eingerichtet ist. Der Test **vom Node selbst** zeigt die Bindung zuverlässig. Ein Zugriff vom Deskmate auf die Node-IP funktioniert nur, wenn die Docker-Netzwerkverbindung dies zulässt. `hostPort` veröffentlicht den Port nicht automatisch auf dem Docker-Host.

Auch Port 80 auf der Node-IP kann bereits von **Traefik** belegt sein. Deshalb wählen wir Port **8080**.

In [ ]:
%%bash
cat network-nginx-hostport.yaml
kubectl apply -f network-nginx-hostport.yaml
kubectl wait --for=condition=Ready pod/network-nginx-hostport --timeout=120s
kubectl get pod network-nginx-hostport -o wide


### 11.1 Den richtigen Node ermitteln

`hostPort` ist an den Node gebunden, **auf dem Pod C tatsächlich läuft**. Wir lesen dessen Node-Namen und InternalIP aus. Der Port ist nicht automatisch auf anderen Nodes verfügbar.

In [ ]:
%%bash
NODE=$(kubectl get pod network-nginx-hostport -o jsonpath='{.spec.nodeName}')
NODE_IP=$(kubectl get node "$NODE" -o jsonpath='{.status.addresses[?(@.type=="InternalIP")].address}')
echo "Pod C läuft auf Node: $NODE"
echo "Node-IP: $NODE_IP"
echo "Ziel: http://$NODE_IP:8080/"


### 11.2 Zugriff auf Node-IP:8080 testen

Zuerst versuchen wir den Zugriff **vom Deskmate**. Ist die Node-IP im Docker-Netzwerk vom Deskmate aus erreichbar, sollte die Antwort **Pod C** erscheinen.

**Falls der Aufruf fehlschlägt:** Das ist bei Docker-basierten K3s-Nodes möglich. `hostPort` bindet an den Kubernetes-Node, nicht automatisch an den Ubuntu-/Docker-Host. Für einen belastbaren Test kann man `docker exec` in den betreffenden Node-Container verwenden, sofern dort ein HTTP-Client vorhanden ist. Der folgende Versuch hat deshalb einen kurzen Timeout und beendet das Notebook bei Nichterreichbarkeit nicht.

In [ ]:
%%bash
NODE=$(kubectl get pod network-nginx-hostport -o jsonpath='{.spec.nodeName}')
NODE_IP=$(kubectl get node "$NODE" -o jsonpath='{.status.addresses[?(@.type=="InternalIP")].address}')
curl -sS --connect-timeout 3 --max-time 6 "http://$NODE_IP:8080/" || echo "Node-IP vom Deskmate nicht erreichbar: Docker-Netzwerk/Portfreigabe prüfen."


### 11.3 Vergleich: `hostPort` ist nicht `containerPort`

Im Manifest von Pod C stehen **beide** Angaben:

```yaml
ports:
  - containerPort: 80
    hostPort: 8080
```

- `containerPort: 80` beschreibt den Port des Containers; allein dadurch wird nichts auf dem Node veröffentlicht.
- `hostPort: 8080` richtet die Weiterleitung vom **ausführenden Node** auf den Container-Port 80 ein.
- Anders als ein **NodePort-Service** gilt diese Veröffentlichung nicht automatisch für alle Nodes.

**Frage:** Was geschieht, wenn ein zweiter Pod auf demselben Node ebenfalls `hostPort: 8080` beansprucht? Er kann dort wegen des Portkonflikts nicht zusätzlich eingeplant werden.

## 12. Zusammenfassung

| Weg | Adresse | Erwartung |
|---|---|---|
| Toolbox → NGINX im selben Pod | `localhost:80` | **Pod A** |
| Toolbox → eigene Pod-IP | `<eigene Pod-IP>:80` | **Pod A** |
| Toolbox → NGINX im anderen Pod | `<andere Pod-IP>:80` | **Pod B** |
| Toolbox → bloßen Pod-Namen | `network-nginx:80` | Ohne Service normalerweise kein DNS-Name |
| Deskmate → Pod | `localhost:18080` via Port-Forwarding | Funktioniert |
| Node-IP → Pod-Port | `<Node-IP>:80` | Ohne Veröffentlichung nicht automatisch |
| Toolbox: zusätzlicher Listener | `9090` | Möglich |
| Toolbox: zweiter Listener | `80` | Portkonflikt mit NGINX |
| Node-IP des ausführenden Nodes → Pod C | `<Node-IP>:8080` | Über `hostPort` möglich, sofern Netzwerk erreichbar |

**DNS-Merksatz:** `/etc/resolv.conf` enthält die DNS-Konfiguration, `/etc/hosts` lokale Namenseinträge. Die Existenz eines Pods erzeugt nicht automatisch einen clusterweit auflösbaren DNS-Namen.

## 13. Aufräumen

Zum Schluss entfernen wir **alle drei Pods**. Es wurden keine Deployments und keine Services für die Übung erstellt.

In [ ]:
%%bash
kubectl delete -f network-multi.yaml
kubectl delete -f network-nginx.yaml
kubectl delete -f network-nginx-hostport.yaml
kubectl get pods
